# Semaine 10 — Jour 7 : Projet NLP intégré et revue des erreurs

Assembler classification, extraction, résumé et NER dans un résultat structuré.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. L’installation de l’environnement ajoute un temps distinct; le lab LoRA est optionnel.

Prérequis : Python 3.11+, bases de classification et de vecteurs. Le parcours utilise NumPy et scikit-learn, sans modèle à télécharger ni clé API. Installer les dépendances S10 selon [le guide](../../../docs/week10/QUICKSTART.md).

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Tenter [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week10/S10_J7_projet_nlp.ipynb` et `notebooks/week10/teacher/S10_J7_projet_nlp_teacher.ipynb`.

Critère de réussite : Livrer artefacts et provenance sans automatiser d’action métier.

Les apprentissages du parcours principal sont réellement exécutables. Le corpus reste fictif, les baselines à règles sont identifiées et aucun résultat LoRA n’est annoncé.


# Objectifs mesurables

- Assembler classification, extraction, résumé et NER dans un résultat structuré.
- Distinguer métriques d’apprentissage, tests logiciels et critères de revue.
- Livrer artefacts et provenance sans automatiser d’action métier.

## Preuves attendues

Une sortie reproductible, un invariant vérifié et une limite de la tâche correctement interprétée.


# S10 J7 — Projet NLP intégré et revue des erreurs


## 1. Ce que reçoit le projet
Une entrée textuelle produit une classe ou une demande de revue, quatre champs extraits avec preuves, un résumé extractif et des spans d’entités. Le résultat ne publie rien et ne modifie aucun compte. Il prépare une analyse que le système applicatif pourrait consommer après validation de ses propres droits et règles.

Les blocs ne sont pas tous appris. La classification et la projection NCA sont entraînées réellement; la tête est adaptée sur un corpus distinct. L’extraction et la NER utilisent des règles, le résumé une centralité TF-IDF. Les désigner séparément permet d’expliquer les erreurs et de choisir une amélioration ciblée.

## 2. Contrat de revue
requires_review vaut vrai lorsque la classification s’abstient, que le ticket_id n’est pas présent ou qu’un champ est ambigu. Un champ optionnel absent ne provoque pas automatiquement une revue. Cette règle est illustrative; elle ne devient pas une autorisation d’agir sur un environnement de production.

Les textes d’exemple ne forment pas un jeu complet de routing. Le classifieur a été entraîné sur des tickets courts; les exemples intégrés sont plus longs et multi-sujets. Inspecter ses sorties au lieu de supposer que les bons résultats de test s’appliquent inchangés à ces nouveaux formats.

## 3. Trois familles de preuve
Les tests logiciels vérifient invariants et cas limites : partitions séparées, offsets, unités, conflits, budget, masquage et rechargement d’artefact. Les métriques de tâche décrivent un comportement sur un corpus : macro-F1, matrice, spans exacts, recouvrement lexical et voisinage thématique. La revue métier détermine l’utilité et les conséquences d’une erreur.

Ces preuves ne sont pas interchangeables. Une suite de tests ne prouve pas la qualité sur tous les tickets; un F1 parfait sur douze exemples ne prouve pas l’absence de bug. Une référence de résumé très proche peut masquer une condition omise. La décision doit garder les trois niveaux visibles.

## 4. Résultats de référence
Dans l’environnement validé, baseline et tête adaptée ont le même F1 de test sur le corpus facile. Les poids changent réellement, mais l’expérience ne démontre pas de gain de classification. Les champs et entités connus sont correctement reconnus sur les quatre exemples annotés, avec une ambiguïté explicitement traitée.

La projection NCA améliore la précision de voisinage par catégorie dans ce petit échantillon. Elle ne mesure pas une pertinence sémantique humaine générale. Les rapports gardent les cas, les ids et les avertissements pour permettre une inspection au-delà des moyennes.

## 5. Artefacts et reproduction
La commande de rapports réentraîne les composants avec la graine fixée, applique les choix de validation puis écrit rapports et classifieurs JSON. Elle enregistre les versions NumPy/scikit-learn/SciPy et les empreintes des données. Le module S10 possède ses dépendances épinglées sans remplacer le pyproject du dépôt.

Les artefacts JSON contiennent les tableaux nécessaires à l’inférence de classification, sans pickle. Leur hash est un contrôle de cohérence local, pas une signature. Pour une livraison réelle, ajouter un stockage/provenance contrôlés et une politique de versions de modèle, données et protocole.

## 6. Prochaine expérience utile
Si les tickets sont nouveaux et ambigus, collecter une annotation représentative avant de choisir un plus grand modèle. Si les noms inconnus manquent, évaluer une NER apprise. Si les formulations courtes ne suffisent pas, analyser le traitement de texte long et ses segments. Si une adaptation LoRA est justifiée, exécuter le lab optionnel avec artefacts locaux et un protocole réservé.

Le projet S11 pourra assembler ces capacités avec API, agents et gouvernance. Cette archive fournit un module autonome et des contrats documentés; elle ne prétend pas avoir branché votre application existante ni déployé un service.

## À retenir
Un projet NLP exploitable sait expliquer ce qu’il apprend, ce qu’il règle, ce qu’il mesure et quand il demande une revue.


## Diagramme du jour

```mermaid
flowchart TD
 A["Ticket texte"] --> B["Classification et abstention"]
 A --> C["Champs et preuves"]
 A --> D["Résumé et entités"]
 B --> E["Résultat structuré"]
 C --> E
 D --> E
 E --> F{"Revue nécessaire ?"}
 F --> G["Analyse, sans action métier"]
 H["Rapports de tâche et tests"] --> I["Décision d’évolution"]
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

NumPy et scikit-learn sont requis; aucune requête réseau dans les cellules.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week10/nlp.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week10.nlp import read_data, process
examples = read_data("examples.json")
results = [{"id":row["id"], **process(row["text"])} for row in examples]
ambiguous = next(row for row in results if row["id"] == "e4")
assert ambiguous["requires_review"]
assert ambiguous["fields"]["http_status"]["status"] == "ambiguous"
print([{ "id":r["id"], "classification":r["classification"]["label"], "review":r["requires_review"]} for r in results])

# Lab J7 — Projet NLP intégré et revue des erreurs

Depuis la racine du dépôt, dans l’environnement S10 :

```bash
python3 book/week10/day07/labs/run.py
python3 -m examples.week10.demo --day 7
```

Résultat attendu : **quatre analyses intégrées, e4 en revue, rapports de tâche distincts**.

## Travail

1. Exécuter le scénario et identifier la nature de chaque composant : appris, adapté ou à règles.
2. Réaliser l’exercice 2 sur une copie, sans modifier le corpus distribué.
3. Expliquer le dénominateur du résultat et une limite.

Les données sont sous `examples/week10/data`; le code commun sous `examples/week10/nlp.py`; les tests sous `tests/week10`. Lire [QUICKSTART](../../../../docs/week10/QUICKSTART.md) pour les dépendances, artefacts et commandes.

Les modèles entraînés sont conservés en mémoire dans chaque processus pour accélérer les notebooks. Une modification des données demande un nouveau processus ou de vider explicitement les caches trained/specialized avant de réentraîner. Les correctifs du notebook enseignant s’exécutent après le lab élève.


# Exercices

Conserver une première réponse avant les corrigés.

## Exercice 1 — Relier les blocs

Quels composants sont appris, adaptés ou à règles ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 2 — Interpréter une revue

Pourquoi un ticket classifié avec forte probabilité peut-il quand même nécessiter une revue ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 3 — Préparer une amélioration

Comment choisir entre enrichir les données et exécuter LoRA ?

Livrable : réponse argumentée avec calcul, sortie ou test.


# Entretien

Répondre en deux minutes par question avec un exemple et une limite.

## Question 1

Comment présenter ce projet sans survendre les scores ?

## Question 2

Que manque-t-il avant une intégration en production ?


# Challenge

Produire les rapports, inspecter le cas ambigu e4 et recharger les deux classifieurs JSON. Rédiger une courte note de décision sur une amélioration prioritaire, en citant une erreur du système plutôt qu’un nom de framework.

## Critères

Protocole reproductible, invariant explicite et interprétation honnête du résultat, y compris sans amélioration.


# Références — Jour 7

Sources primaires consultées le 20 octobre 2026. Le corpus et les annotations du lab sont fictifs; les seuils sont pédagogiques.

- [scikit-learn 1.8 : pièges et fuite de données](https://scikit-learn.org/1.8/common_pitfalls.html).
- [SGDClassifier 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.SGDClassifier.html) — log_loss et partial_fit.
- [TfidfVectorizer 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).
- [Neighborhood Components Analysis 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.neighbors.NeighborhoodComponentsAnalysis.html) — apprentissage métrique supervisé.
- [spaCy : entités nommées](https://spacy.io/usage/linguistic-features#named-entities) — vocabulaire et conventions; spaCy n’est pas utilisé dans le lab obligatoire.
- [PEFT : LoRA](https://huggingface.co/docs/peft/package_reference/lora) — extension Transformer optionnelle, non exécutée.

Lire en priorité les références du thème étudié. Les concepts communs à plusieurs bibliothèques ne rendent pas leurs objets ou artefacts interchangeables. Les dépendances obligatoires correspondent à l’environnement testé, les dépendances LoRA restent à verrouiller pour l’expérience optionnelle.
